# Beach kiosk sales and weather: correlations

**Correlation and Partial Correlation Matrix** · a worked example from the StatNex teaching guide

**The story.** A beach kiosk logged twelve months of temperature, visitors, rainy days and sales of ice cream, sunscreen and hot coffee. The owner noticed that ice cream and sunscreen sales rise and fall together and wonders whether to bundle them.

**The question.** Do ice cream sales really drive sunscreen sales, or are both just following the weather?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `01_basic/correlation_matrix.csv` (12 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Month,Temperature (C),Visitors (thousands),Ice cream sales ($000),Sunscreen sales ($000),Hot coffee sales ($000),Rainy days
Jan,4.2,1.7,4.1,1.6,23.6,11.0
Feb,5.0,2.0,4.8,1.5,22.3,10.0
Mar,9.2,10.4,7.2,2.1,20.3,11.0
Apr,14.6,12.9,13.1,5.4,16.5,9.0
May,19.8,18.2,15.5,9.7,11.6,6.0
Jun,22.6,22.3,16.5,10.6,12.4,8.0
Jul,27.4,22.7,19.9,11.7,8.0,7.0
Aug,26.3,25.3,20.8,11.4,10.0,5.0
Sep,24.5,20.4,16.9,11.4,9.4,6.0
Oct,18.9,16.0,14.9,8.9,13.9,8.0
Nov,10.2,9.3,9.8,4.0,19.6,11.0
Dec,7.0,4.8,4.7,3.6,19.4,12.0

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 160)

## The data

In [ ]:
# One column per variable, one row per case. Blank and text cells are missing.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Temperature (C)", "Visitors (thousands)", "Ice cream sales ($000)", "Sunscreen sales ($000)", "Hot coffee sales ($000)", "Rainy days"]
numeric = data[columns].apply(pd.to_numeric, errors="coerce")
# Listwise deletion: a case missing any of the variables is dropped from all of them.
numeric = numeric.dropna()
counts = pd.DataFrame(len(numeric), index=columns, columns=columns)

## Correlation matrix

In [ ]:
# Pearson's r for every pair: how closely the two variables follow a straight line,
# from -1 (falling line) through 0 (no straight-line link) to +1 (rising line). It
# measures association only, never which variable drives which.
corr = numeric.corr(method="pearson")
print(corr)

## Test of zero correlation

In [ ]:
# scipy.stats.pearsonr() for each pair: t = r √(n - 2) / √(1 - r²) on n - 2 df,
# two-sided. A small P says a straight-line link this strong would be rare if the
# true r were 0; it says nothing about how strong the link is. It assumes independent
# cases (months in a row often are not) and roughly normal scatter. Each pair uses
# the cases observed on both, which is pairwise deletion when the data still have
# gaps. The app prints the P value above the diagonal and * (P < 0.05) / ** (P < 0.01)
# below it.
r = corr.to_numpy()
n = counts.to_numpy(float)
pairs = [(i, j) for i in range(len(columns)) for j in range(i + 1, len(columns))]
tests = {}
p_values = np.full((len(columns), len(columns)), np.nan)
for i, j in pairs:
    if n[i, j] > 2 and abs(r[i, j]) < 1:
        both = numeric[[columns[i], columns[j]]].dropna()
        tests[i, j] = stats.pearsonr(both.iloc[:, 0], both.iloc[:, 1])
        p_values[i, j] = p_values[j, i] = tests[i, j].pvalue
print()
print(pd.DataFrame(p_values, index=columns, columns=columns))
# Every pair once, in the order of the matrix's upper triangle: the list `pairs`
# above.

## Partial correlation matrix

In [ ]:
# The correlation of each pair with all the other variables held fixed: the link
# left between two variables once the straight-line effect of every other variable
# is taken out of both. A large r with a small partial r points to a common driver.
# scipy and statsmodels have no partial-correlation matrix, so it is read off the
# inverse of the correlation matrix: -inverse[i, j] / √(inverse[i, i] inverse[j, j]).
inverse = np.linalg.inv(r)
scale = np.sqrt(np.diag(inverse))
partial = -inverse / np.outer(scale, scale)
np.fill_diagonal(partial, 1.0)
partial = pd.DataFrame(partial, index=columns, columns=columns)
print()
print(partial)

## Sample size

In [ ]:
print()
print(counts)

## What the result says

A correlation runs from -1 to 1 and shows how closely two measures move together. Ice cream and sunscreen sales have a correlation of 0.96, so they rise and fall almost in step. But both also follow temperature very closely: 0.99 for ice cream and 0.98 for sunscreen.

A partial correlation is the link between two measures after the other measures are held constant. For ice cream and sunscreen it is only -0.18. Once temperature, visitors, coffee and rain are taken out, almost no link is left.

So ice cream does not drive sunscreen. Both follow the weather. With only 12 months this is a strong hint, not proof, and it gives no reason to expect that a bundle would lift sales.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.